In [1]:
#@title Download librerie
!pip install torch transformers accelerate datasets numpy tqdm scikit-learn wandb

In [2]:
#@title Importazione moduli
!pip install -U datasets
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForMaskedLM, TrainingArguments, Trainer

Requirement already up-to-date: datasets in /home/johnnyfreak/.local/lib/python3.8/site-packages (3.1.0)


In [3]:
#@title Importazione del modello da **Huggingface**
# model = AutoModelForMaskedLM.from_pretrained("cabrooks/LOGION-50k_wordpiece")
model = AutoModelForMaskedLM.from_pretrained('bowphs/PhilBerta')

model.num_parameters() / 1_000_000

135.259648

In [4]:
# @title Importazione del tokenizer del modello
# tokenizer = AutoTokenizer.from_pretrained("cabrooks/LOGION-50k_wordpiece")
tokenizer = AutoTokenizer.from_pretrained('bowphs/PhilBerta')
tokenizer.model_max_length = 512

tokenizer_config.json:   0%|          | 0.00/354 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [5]:
#@title Importazione del dataset
from datasets import load_dataset, DatasetDict

dataset = load_dataset("CNR-ILC/gs-maat-corpus")


In [6]:
#@title Importazione del *data collator*
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [7]:
#@title Tokenizzazione dei dati e raggruppamento in *chunk*
from itertools import chain

desired_columns = ['input_ids', 'attention_mask', 'labels'] 

def tokenize_function(examples):
    return tokenizer(examples["text"], padding=False, add_special_tokens=False)

chunk_size = 128
def group_texts(examples):
    # Prima si concatenano tutti i token
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    
    total_length = len(concatenated_examples["input_ids"])
    # Padding fino al prossimo chunk_size
    if total_length % chunk_size != 0:
        padding_length = chunk_size - (total_length % chunk_size)
        for key in concatenated_examples.keys():
            if key == "attention_mask":
                pad_value = 0
            else:
                pad_value = tokenizer.pad_token_id
            concatenated_examples[key] += [pad_value] * padding_length

    total_length = (len(concatenated_examples["input_ids"]) // chunk_size) * chunk_size
    # Spezza in chunk
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

def process_data(examples):
    tokenized_output = tokenize_function(examples)
    return group_texts(tokenized_output)

lm_datasets = dataset.map(process_data, batched=True, remove_columns=['text'])
lm_datasets = lm_datasets.select_columns(desired_columns)

Map:   0%|          | 0/84751 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (627 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/9303 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [8]:
#@title TopK accuracy
import torch
import numpy as np
from sklearn.metrics import top_k_accuracy_score

def compute_metrics(eval_pred, compute_result=False):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    
    # Conversione sicura da torch (standard con accelerate) -> numpy (per calcolo top_k_accuracy_score) 
    if isinstance(logits, torch.Tensor):
        logits = logits.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()

    # Reshape di tutto per lavorare a livello dei singoli token
    logits = logits.reshape(-1, logits.shape[-1])    # (batch_size * sequence_length, vocab_size)
    labels = labels.flatten()                        # (batch_size * sequence_length)

    # Ora possiamo lavorare correttamente
    preds = np.argmax(logits, axis=1)
    
    # Solo per i token non mascherati (-100 è il valore ignorato nei label di Huggingface)
    mask = labels != -100
    if not np.any(mask):
        return {'top1_acc': 0,
        'top5_acc': 0,
        'top10_acc': 0,
        'top15_acc': 0,
        'top20_acc': 0,
        'top25_acc': 0,
        'top30_acc': 0,
        'top35_acc': 0,
        'top40_acc': 0,
        'top45_acc': 0,
        'top50_acc':0}
    
    preds = preds[mask]
    labels = labels[mask]
    logits = logits[mask]
    
    return {
        'top1_acc': (preds == labels).mean(),
        'top5_acc': top_k_accuracy_score(labels, logits, k=5, labels=np.arange(logits.shape[1])),
        'top10_acc': top_k_accuracy_score(labels, logits, k=10, labels=np.arange(logits.shape[1])),
        'top15_acc': top_k_accuracy_score(labels, logits, k=15, labels=np.arange(logits.shape[1])),
        'top20_acc': top_k_accuracy_score(labels, logits, k=20, labels=np.arange(logits.shape[1])),
        'top25_acc': top_k_accuracy_score(labels, logits, k=25, labels=np.arange(logits.shape[1])),
        'top30_acc': top_k_accuracy_score(labels, logits, k=30, labels=np.arange(logits.shape[1])),
        'top35_acc': top_k_accuracy_score(labels, logits, k=35, labels=np.arange(logits.shape[1])),
        'top40_acc': top_k_accuracy_score(labels, logits, k=40, labels=np.arange(logits.shape[1])),
        'top45_acc': top_k_accuracy_score(labels, logits, k=45, labels=np.arange(logits.shape[1])),
        'top50_acc': top_k_accuracy_score(labels, logits, k=50, labels=np.arange(logits.shape[1])),
    }

In [9]:
#@title Definizione dei parametri di addestramento
import torch
from transformers import TrainingArguments
torch.cuda.empty_cache() #Pulisco la cache della GPU per liberare memoria prima del finetuning
training_args = TrainingArguments(
    output_dir="./gs-greBERTa",
    logging_dir="./greBERTa-logs", 
    report_to="wandb",         #weight & biases per il report delle metriche durante il finetuning
    eval_strategy="epoch",     # Valutazione basata sulle epoche
    save_strategy="no", 
    learning_rate=5e-5, # Passo     
    per_device_train_batch_size=16,
    per_device_eval_batch_size=1,
    num_train_epochs=10,             # Numero di epoche
    seed=42,                         # Seme per la riproducibilità
    fp16=True,                       # Precisione mista per migliorare le prestazioni
    logging_strategy="epoch",        # Log basati sulle epoche
    eval_accumulation_steps=1,       # Numero di batch che vengono elaborati sulla GPU prima di spostarli sulla CPU 
    batch_eval_metrics=True,         # Chiama `compute_metrics`, permettendo di non mantenere tutto in memoria 
    torch_empty_cache_steps=5000,     #Pulisce la cache ogni 5000 passi, evita problemi di OOM 
    dataloader_drop_last=False,       # Evita batch incompleti in validazione
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [10]:
lm_datasets["test"][0]

{'input_ids': [8421,
  1828,
  24084,
  8949,
  3017,
  2181,
  19274,
  11048,
  6365,
  1118,
  11048,
  1706,
  1302,
  1706,
  836,
  19521,
  49272,
  1706,
  1587,
  28231,
  1477,
  22189,
  3444,
  1477,
  11048,
  63749,
  3017,
  1812,
  22154,
  2199,
  28231,
  25143,
  1888,
  41452,
  3090,
  49845,
  2312,
  3721,
  3713,
  1812,
  1706,
  32497,
  1282,
  8150,
  49272,
  1706,
  799,
  24084,
  57842,
  22726,
  19274,
  10241,
  3721,
  21176,
  8150,
  34728,
  34057,
  1320,
  24084,
  2199,
  30023,
  1118,
  19994,
  34057,
  836,
  10241,
  49845,
  2312,
  3721,
  3713,
  1812,
  1706,
  32497,
  1282,
  8150,
  49272,
  1706,
  799,
  24084,
  57842,
  22726,
  19274,
  10241,
  3721,
  21176,
  8150,
  34728,
  34057,
  1320,
  24084,
  2199,
  30023,
  1118,
  19994,
  34057,
  836,
  10241,
  49845,
  1380,
  21928,
  34728,
  1302,
  22189,
  1524,
  1477,
  4696,
  61549,
  1118,
  8150,
  21176,
  3017,
  49845,
  43876,
  3090,
  13033,
  4474,
  2199,
 

In [11]:
#@title finetuning (*continual-pretraining*) del modello
import math
import json 
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=lm_datasets["dev"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
) 

results = trainer.evaluate(eval_dataset=lm_datasets["test"])

#with open ("eval_results_Logion_normal.json", "w") as f: 
with open ("eval_results_PhilBERTA.json", "w") as f: 
     json.dump(results, f, indent=4)
#eval_results = trainer.evaluate(eval_dataset=lm_datasets["test"])
#print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Currently logged in as: s-zenzaro (s-zenzaro-cnr-ilc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
hugg